In [ ]:

import os
from pathlib import Path
from llama_index import SimpleDirectoryReader, GPTVectorStoreIndex, LLMPredictor, ServiceContext
from llama_index.llms import OpenAI as LlamaOpenAI
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import Chroma
from langchain.chains import RetrievalQA
from langchain.chat_models import ChatOpenAI

DATA_DIR = "data_txt"   # text files created by Adobe Extract/preprocessing
INDEX_DIR = "llama_index_store"
PERSIST_DIR = "./miracl_chroma_hybrid"
OPENAI_KEY = os.getenv("OPENAI_API_KEY")

# 1) Build index with LlamaIndex 
def build_llamaindex():
    llm = LlamaOpenAI(api_key=OPENAI_KEY, model="gpt-4o-mini")
    predictor = LLMPredictor(llm=llm)
    service_context = ServiceContext.from_defaults(llm_predictor=predictor)
    docs = SimpleDirectoryReader(DATA_DIR).load_data()
    index = GPTVectorStoreIndex.from_documents(docs, service_context=service_context)
    index.storage_context.persist(persist_dir=INDEX_DIR)
    print("LlamaIndex built.")
    return index

# 2) Use LangChain to create a Retriever from Chroma (same persisted store) and run RetrievalQA
def langchain_query_loop():
    embedder = OpenAIEmbeddings(openai_api_key=OPENAI_KEY)
    db = Chroma(persist_directory=PERSIST_DIR, embedding_function=embedder)  
    retriever = db.as_retriever(search_kwargs={"k":3})
    llm = ChatOpenAI(model_name="gpt-4o-mini", openai_api_key=OPENAI_KEY, temperature=0)
    qa = RetrievalQA.from_chain_type(llm=llm, chain_type="stuff", retriever=retriever)
    while True:
        q = input("Q> ")
        if q.lower() in ("exit","quit"): break
        print(qa.run(q))

if __name__=="__main__":
    build_llamaindex()
    print("Now run LangChain query loop (ensure persistence paths match).")
    langchain_query_loop()
